# 📝 Text Classification: RNN vs. Transformer

**CO5085 – Assignment 1**

So sánh: **BiLSTM, GRU** vs. **DistilBERT**

In [ ]:
import sys, torch, os, json
sys.path.insert(0, '../src')
os.makedirs('../results', exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

TRAIN_MODE = False  # False = load checkpoint (nhanh) | True = train lại từ đầu
print(f'Device: {DEVICE} | Train mode: {TRAIN_MODE}')

## 1. Load Data (20 Newsgroups + DistilBERT Tokenizer)

In [ ]:
from datasets import get_20newsgroups_loaders
train_loader, val_loader, test_loader, tokenizer, num_classes = get_20newsgroups_loaders(
    tokenizer_name='distilbert-base-uncased', batch_size=32, max_length=256)
print(f'Classes: {num_classes} | Train batches: {len(train_loader)}')

## 2. Model 1: DistilBERT (Transformer)

In [ ]:
from models import get_distilbert

distilbert = get_distilbert(num_classes=num_classes)
if TRAIN_MODE:
    from train import train
    history_distilbert = train(distilbert, train_loader, val_loader,
                               num_epochs=3, lr=2e-5, device=DEVICE,
                               save_path='../results/distilbert_best.pt', scheduler_type='cosine')
else:
    distilbert.load_state_dict(torch.load('../results/distilbert_best.pt', map_location=DEVICE))
    with open('../results/distilbert_history.json') as f: history_distilbert = json.load(f)
    print('DistilBERT loaded ✅')
distilbert = distilbert.to(DEVICE).eval()

## 3. BiLSTM (RNN)

> **Note**: BiLSTM dùng cùng DistilBERT tokenizer (vocab_size=30522) như training script.

In [ ]:
from models import BiLSTMClassifier

if TRAIN_MODE:
    from train import train
    bilstm = BiLSTMClassifier(vocab_size=tokenizer.vocab_size, num_classes=num_classes)
    history_bilstm = train(bilstm, train_loader, val_loader,
                           num_epochs=5, lr=1e-3, device=DEVICE,
                           save_path='../results/bilstm_best.pt')
else:
    ckpt = torch.load('../results/bilstm_best.pt', map_location='cpu')
    vocab_size_bilstm = ckpt['embedding.weight'].shape[0]
    embed_dim_bilstm  = ckpt['embedding.weight'].shape[1]
    bilstm = BiLSTMClassifier(vocab_size=vocab_size_bilstm, embed_dim=embed_dim_bilstm, num_classes=num_classes)
    bilstm.load_state_dict(ckpt)
    with open('../results/bilstm_history.json') as f: history_bilstm = json.load(f)
    print(f'BiLSTM loaded ✅ (vocab={vocab_size_bilstm}, embed={embed_dim_bilstm})')
bilstm = bilstm.to(DEVICE).eval()

## 4. GRU (RNN)

In [ ]:
from models import GRUClassifier

if TRAIN_MODE:
    from train import train
    gru = GRUClassifier(vocab_size=tokenizer.vocab_size, num_classes=num_classes)
    history_gru = train(gru, train_loader, val_loader,
                        num_epochs=5, lr=1e-3, device=DEVICE,
                        save_path='../results/gru_best.pt')
else:
    ckpt = torch.load('../results/gru_best.pt', map_location='cpu')
    vocab_size_gru = ckpt['embedding.weight'].shape[0]
    embed_dim_gru  = ckpt['embedding.weight'].shape[1]
    gru = GRUClassifier(vocab_size=vocab_size_gru, embed_dim=embed_dim_gru, num_classes=num_classes)
    gru.load_state_dict(ckpt)
    with open('../results/gru_history.json') as f: history_gru = json.load(f)
    print(f'GRU loaded ✅ (vocab={vocab_size_gru}, embed={embed_dim_gru})')
gru = gru.to(DEVICE).eval()

## 5. Evaluation & Comparison

In [ ]:
from evaluate import compute_metrics, plot_training_curves, compare_models, get_predictions
import numpy as np

results_text = {}
for model_name, model, history in [
    ('DistilBERT', distilbert, history_distilbert),
    ('BiLSTM',     bilstm,     history_bilstm),
    ('GRU',        gru,        history_gru),
]:
    print(f'\n=== {model_name} ===')
    preds, labels, _ = get_predictions(model, test_loader, DEVICE)
    results_text[model_name] = compute_metrics(preds, labels, verbose=True)
    plot_training_curves(history, model_name, f'../results/{model_name.lower()}_curves.png')

compare_models(results_text, metric='accuracy', save_path='../results/text_comparison_acc.png')
compare_models(results_text, metric='f1_macro',  save_path='../results/text_comparison_f1.png')